In [0]:
-- 1. DATA QUALITY CHECK
----------------------------------------
-- Data Preview
SELECT * 
FROM `brightlearn`.`case_study`.`bright_motor`
LIMIT 10;

-- Check distinct values
SELECT DISTINCT make, body, transmission, state
FROM `brightlearn`.`case_study`.`bright_motor`;

-- Check duplicate VINs
SELECT 
    vin,
    COUNT(*) AS duplicate_count
FROM `brightlearn`.`case_study`.`bright_motor`
GROUP BY vin
HAVING COUNT(*) > 1;

-- Check missing important fields
SELECT *
FROM `brightlearn`.`case_study`.`bright_motor`
WHERE sellingprice IS NULL 
   OR make IS NULL 
   OR model IS NULL;


-- 2. DATA CLEANING & FINAL TABLE
----------------------------------------

CREATE OR REPLACE TABLE `brightlearn`.`case_study`.`bright_motor_clean` AS
SELECT

    -- Core fields
    year,
    COALESCE(make, 'Unknown') AS make,
    COALESCE(model, 'Unknown') AS model,
    COALESCE(trim, 'Unknown') AS trim,
    COALESCE(body, 'Unknown') AS body,
    COALESCE(transmission, 'Unknown') AS transmission,
    vin,
    COALESCE(state, 'Unknown') AS state,

    condition,
    odometer,
    COALESCE(seller, 'Unknown') AS seller,

    -- Numeric conversions (simple + safe)
    CAST(NULLIF(sellingprice, '') AS FLOAT64) AS selling_price,
    CAST(NULLIF(mmr, '') AS FLOAT64) AS mmr,

    -- Date conversion
    DATE(saledate) AS sale_date,

    -- Time breakdown
    EXTRACT(YEAR FROM DATE(saledate)) AS sale_year,
    EXTRACT(MONTH FROM DATE(saledate)) AS sale_month,

    -- Car age
    EXTRACT(YEAR FROM CURRENT_DATE()) - year AS car_age,

    -- Revenue
    CAST(NULLIF(sellingprice, '') AS FLOAT64) AS revenue,

    -- Profit
    CAST(NULLIF(sellingprice, '') AS FLOAT64) 
    - CAST(NULLIF(mmr, '') AS FLOAT64) AS profit,

    -- Profit margin
    (
        (CAST(NULLIF(sellingprice, '') AS FLOAT64) 
        - CAST(NULLIF(mmr, '') AS FLOAT64))
        / CAST(NULLIF(sellingprice, '') AS FLOAT64)
    ) * 100 AS profit_margin,

    -- Profit tier
    CASE 
        WHEN ((sellingprice - mmr) / sellingprice) * 100 >= 20 THEN 'High'
        WHEN ((sellingprice - mmr) / sellingprice) * 100 >= 10 THEN 'Medium'
        ELSE 'Low'
    END AS profit_tier,

    -- Mileage grouping
    CASE 
        WHEN odometer < 20000 THEN 'Low Mileage'
        WHEN odometer < 80000 THEN 'Medium Mileage'
        ELSE 'High Mileage'
    END AS mileage_band,

    -- Price vs market
    CASE 
        WHEN sellingprice > mmr THEN 'Above Market'
        WHEN sellingprice < mmr THEN 'Below Market'
        ELSE 'At Market'
    END AS price_vs_market

FROM `brightlearn`.`case_study`.`bright_motor`;


-- 3. DATA OVERVIEW
----------------------------------------

SELECT COUNT(*) AS total_sales 
FROM `brightlearn`.`case_study`.`bright_motor_clean`;

SELECT COUNT(DISTINCT make) AS total_brands 
FROM `brightlearn`.`case_study`.`bright_motor_clean`;

SELECT COUNT(DISTINCT model) AS total_models 
FROM `brightlearn`.`case_study`.`bright_motor_clean`;


-- 4. REVENUE ANALYSIS
----------------------------------------

-- Revenue by Make
SELECT 
    make, 
    SUM(revenue) AS total_revenue
FROM `brightlearn`.`case_study`.`bright_motor_clean`
GROUP BY make
ORDER BY total_revenue DESC;

-- Top Models
SELECT 
    make, 
    model, 
    COUNT(*) AS cars_sold
FROM `brightlearn`.`case_study`.`bright_motor_clean`
GROUP BY make, model
ORDER BY cars_sold DESC;


-- 5. PRODUCT ANALYSIS
----------------------------------------

-- Body Type Preference
SELECT 
    body, 
    COUNT(*) AS total_sales
FROM `brightlearn`.`case_study`.`bright_motor_clean`
GROUP BY body
ORDER BY total_sales DESC;

-- Transmission Preference
SELECT 
    transmission, 
    COUNT(*) AS total_sales
FROM `brightlearn`.`case_study`.`bright_motor_clean`
GROUP BY transmission;

-- Car Age vs Price
SELECT 
    car_age, 
    AVG(selling_price) AS avg_price
FROM `brightlearn`.`case_study`.`bright_motor_clean`
GROUP BY car_age
ORDER BY car_age;


-- 6. REGIONAL ANALYSIS
----------------------------------------

SELECT 
    state, 
    COUNT(*) AS cars_sold,
    SUM(revenue) AS total_revenue
FROM `brightlearn`.`case_study`.`bright_motor_clean`
GROUP BY state
ORDER BY total_revenue DESC;


-- 7. SALES TREND ANALYSIS
----------------------------------------

-- Yearly sales
SELECT 
    sale_year, 
    COUNT(*) AS cars_sold
FROM `brightlearn`.`case_study`.`bright_motor_clean`
GROUP BY sale_year
ORDER BY sale_year;

-- Monthly sales
SELECT 
    sale_month, 
    COUNT(*) AS cars_sold
FROM `brightlearn`.`case_study`.`bright_motor_clean`
GROUP BY sale_month
ORDER BY sale_month;


-- 8. PRICING & PROFITABILITY ANALYSIS
----------------------------------------

-- Profit margin by make
SELECT 
    make, 
    AVG(profit_margin) AS avg_margin
FROM `brightlearn`.`case_study`.`bright_motor_clean`
GROUP BY make
ORDER BY avg_margin DESC;

-- Price vs market
SELECT 
    price_vs_market, 
    COUNT(*) AS total_sales
FROM `brightlearn`.`case_study`.`bright_motor_clean`
GROUP BY price_vs_market;


-- 9. SELLER PERFORMANCE
----------------------------------------

SELECT 
    seller,
    COUNT(*) AS cars_sold,
    SUM(revenue) AS total_revenue
FROM `brightlearn`.`case_study`.`bright_motor_clean`
GROUP BY seller
ORDER BY total_revenue DESC;


-- 10. ADVANCED INSIGHTS
----------------------------------------

-- Mileage vs price
SELECT 
    mileage_band,
    AVG(selling_price) AS avg_price
FROM `brightlearn`.`case_study`.`bright_motor_clean`
GROUP BY mileage_band;

-- Most profitable models
SELECT 
    make, 
    model,
    AVG(profit) AS avg_profit
FROM `brightlearn`.`case_study`.`bright_motor_clean`
GROUP BY make, model
ORDER BY avg_profit DESC;

-- Profit tier distribution
SELECT 
    profit_tier, 
    COUNT(*) AS total_sales
FROM `brightlearn`.`case_study`.`bright_motor_clean`
GROUP BY profit_tier;